In [0]:
from pyspark.sql import functions as F
from datetime import datetime

# SETUP REQUIRED: Serverless compute requires Unity Catalog Volumes instead of /FileStore/
# 
# Steps to fix:
# 1. Create a volume: CREATE VOLUME IF NOT EXISTS workspace.bronze.data_source;
# 2. Upload your CSV files to: /Volumes/workspace/bronze/data_source/
# 3. The files should be: events.csv, organizations.csv, ticket_orders.csv, etc.
#
# Once complete, this path will work:
seed_path = "/Volumes/workspace/bronze/data_source/"

In [0]:
# Mapeamento de tabelas usando tuplas
SEED_TABLES = [
    ("events", "bronze.events"),
    ("event_tags", "bronze.event_tags"),
    ("event_tag_map", "bronze.event_tag_map"),
    ("event_roles", "bronze.event_roles"),
    ("event_staff", "bronze.event_staff"),
    ("organizations", "bronze.organizations"),
    ("ticket_orders", "bronze.ticket_orders"),
    ("ticket_order_items", "bronze.ticket_order_items"),
    ("sale_transactions", "bronze.sales_transactions"),
    ("sale_items", "bronze.sales_items"),
    ("subscriptions", "bronze.subscriptions"),
    ("subscription_plans", "bronze.subscription_plans"),
    ("products", "bronze.products"),
    ("points_of_sale", "bronze.points_of_sale"),
    ("event_inventory", "bronze.event_inventory"),
    ("inventory_movements", "bronze.inventory_movements"),
    ("budgets", "bronze.budgets"),
    ("budget_categories", "bronze.budget_categories"),
    ("expenses", "bronze.expenses"),
    ("users", "bronze.users")
]

for csv_name, table_name in SEED_TABLES:
    df = (spark.read
          .option("header", "true")
          .option("inferSchema", "true")
          .csv(f"{seed_path}{csv_name}.csv"))
    df = df.withColumn("_ingested_at", F.lit(datetime.now()))

    (df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name)
    )

    count = df.count()
    print(f"{table_name}: {count:,} registros ingeridos")